# AgrMap 

**Developer:** [Camilla Lo Giudice](https://github.com/Camilogiu) <br />
**Contributor:** [Ariane Millot](https://github.com/ariane-millot) <br />

---------------------------

This notebook uses the functions in BaseMap.py to generate a calibrated, historical, geospatially explicit agricultural map. The workflow produces a point layer containing harvested crop areas. Each crop is further divided into four main farming systems, consistent with the GAEZ classification: high-irrigated, high-rainfed, low-irrigated, and low-rainfed.

Before running the notebook, ensure that all required input data listed in the README file are prepared, run the FarmlandClassification.ipynb, and fill the User Settings cell in this notebook.


### Import Packages

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import rasterio
import requests
import geopandas as gpd
from shapely.geometry import Point
from rasterio.features import rasterize
from pyproj import CRS
import time
notebook_start = time.perf_counter()


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Map.BaseMap import Raster, Map, Water 



### User Settings

In [ ]:
Country_name = 'ZMB'                  #Country ISO code
FAOSTAT_country_name = 'Zambia'       #The name has to be the same as in the FAOSTAT dataset. If you are unsure, please check the FAOSTAT_HarvestedArea2020.csv files under the column "Area"
cropland_value = (10, 11, 20)              # specify the pixel value of cropland; 

#Calibration can be done using the FAOSTAT harvested area data, or national statistics, if available. 
#Choose among FAO or National to calibrate the cropland area. If you choose national, please provide the harvested area in the format of the lookup table.



Calibration_method = 2              # Choose among 0 if adm0, 1 if adm1, 2 if adm2

#SSP and climate model selection are needed for extracting crop water deficit data from the GAEZ v5 database (Step 9). If you are not interested in this step, you can ignore these two parameters.
 
SSP = "historical"                    # Choose your preferred SSP (Shared Socioeconomic Pathway) from the available options: historical, ssp126, ssp370, ssp585.
Climate_model = "AGERA5"              #If ssp historical the only climate model is AGERA5. For all the other SSPs, the available climate models are: GFDL-ESM4, IPSL-CM6A-LR, MPI-ESM1-2-HR, MRI-ESM2-0, UKESM1-0-LL.


## Coordinate and projection systems. The CRS must be in meters.

crs_WGS84 = CRS("EPSG:4326")          # For the analysis, the original WGS84 coordinate system is used. 
crs_proj = CRS("EPSG:32735")          # Please provide proper projection system based on the geographical location of the selected country. 
                                      # More info: http://epsg.io/ 

### Directories

In [ ]:
ROOT_DIR = os.path.abspath(os.curdir)
OUT_NODATA = 255
Data_dir = os.path.join(ROOT_DIR, 'Data')
Input_data = os.path.join(Data_dir, '00.InputData')

land_cover_dir = os.path.join(Input_data, 'Land Cover')
National_statistics = os.path.join(Input_data, "FAOSTAT_HarvestedArea2023.csv") #update the name if you change the file

### Crop calibration feasibility check 
This function checks if the total harvested area exceedes the available pixel area. For each cropland_value, the feasibility check prints the pixel count/area, compares the total cropland area in the raster against the national FAOSTAT harvested area, and warns
if the raster doesn't have enough cropland capacity to hold it (a prerequisite for Step 7 calibration/redistribution to succeed without raising an error).

In [ ]:
# Validate cropland pixel count vs National harvested area
validation = Raster.validate_cropland_feasibility(
    Country_name=Country_name,
    FAOSTAT_country_name=FAOSTAT_country_name,
    cropland_value=cropland_value,
    harvested_csv=National_statistics,
)


### Step 1: Create Base Cropland Layer

In [ ]:
# Reproject the land cover raster to the target projection system (crs_proj).
reprojected_cropland_raster = Raster.reproject_rasters(
    Country_name=Country_name,
    input_dir= land_cover_dir,
    output_dir= Input_data,
    out_nodata=OUT_NODATA,
    target_crs=crs_proj, 
)   

In [ ]:
land_cover_raster = os.path.join(Input_data, f'{Country_name}_LandCover.tif')
output_cropland_raster = Raster.reclassify_land_cover(
    Country_name = Country_name,
    input_raster= land_cover_raster,
    output_dir=None,
    cropland_value=cropland_value,
    out_nodata=OUT_NODATA
)

In [ ]:
#converting all data that are not cropland into NoData. 

nodata_cropland_raster = os.path.splitext(output_cropland_raster)[0] + "_nod.tif"

with rasterio.open(output_cropland_raster) as src:
        profile = src.profile.copy()
        data = src.read(1)
        data = np.where(data == 0, OUT_NODATA, data)
        profile.update(nodata=OUT_NODATA)

with rasterio.open(nodata_cropland_raster, 'w', **profile) as dst:
        dst.write(data, 1)

In [ ]:
# #Run if the cell below returns an error. This is to check the fiona version and path.
# #If it returns "None" then run conda install -c conda-forge fiona geopandas

# import fiona
# print(fiona.__file__)
# print(getattr(fiona, "__version__", None))

In [ ]:
# Convert the cropland raster to points vector with coordinates
output_points_dir = Raster.raster_to_points(
    Country_name = Country_name,
    input_tif=None,
    output_dir=None,
    value_column='cropland'
)

In [ ]:
# Reproject and filter subnational administrative boundaries to the target projection system 
if Calibration_method > 0: 
    output_adm_boundaries = Raster.create_subnational_shapefile(Country_name=Country_name)
else: 
    pass


In [ ]:
# Rasterize sub-national admin boundaries (province/district) onto the same grid as the
# reclassified cropland raster, so points can later be tagged with adm1_name/adm2_name.
if Calibration_method > 0:
    admin_raster_fp = Raster.rasterize_admin_boundaries(
        Country_name=Country_name,
    )
    print("Admin raster:", admin_raster_fp)
else: 
    pass

### Step 2: Add Farmland Information 

Farmland information is needed to classify high-mechanized farming. Farmland shapefiles can be downloaded from Open Street Map (OSM). However, we need to create consistency across the farming systems' naming convention used in OSM and by the IFPRI dataset.

**Before running this cell** go to the FarmlandClassification Jupyter Notebook to get the correct shapefile. Follow the instructions to download OSM data and re-classify them according to the context. 

This cell will then rasterize the reclassified farmland polygons to the same grid as the reference land cover raster,then sample those values on the cropland GeoPackage and add a "farms" column.

Missing/no-farm pixels will have "NoFarm" in the 'farms' column.

In [ ]:
farmland_raster, farmland_gpkg = Raster.rasterize_farmlands(
    Country_name=Country_name,
    output_dir=None,
    out_nodata=OUT_NODATA,
    admin_level=Calibration_method
)

### Step 3 – Process MAPSPAM Crop Rasters

Step 3 clips MAPSPAM rasters based on country boundaries and reproject them to the user defined coordinate system

In [ ]:
spam_input_dir = os.path.join(Input_data, "spam2020V2r2_global_harvested_area")
country_boundary = os.path.join(Input_data, f"gadm41_{Country_name}_0.shp")

spam_clip_dir, clipped_files = Raster.clip_rasters(
    Country_name=Country_name,
    input_dir=spam_input_dir,
    boundary_path=country_boundary,
)

print("Clipped rasters:", spam_clip_dir)

In [ ]:
spam_clip_dir = os.path.join(Data_dir, f"02.{Country_name}_MAPSPAM_Crops", f"01.{Country_name}_Clipped_Rasters" )
reprojected_dir = os.path.join(Data_dir, f"02.{Country_name}_MAPSPAM_Crops", "02.Reprojected_Rasters")
spam_reproj_dir, reprojected_files = Raster.reproject_rasters(
    Country_name=Country_name,
    input_dir=spam_clip_dir,
    output_dir=reprojected_dir,
    out_nodata=OUT_NODATA,
    target_crs=crs_proj, 
)

print("Reprojected rasters:", spam_reproj_dir)

### Step 4 – Calculate the share of irrigated area and remove empty rasters 

In [ ]:
spam_reproj_dir = os.path.join(Data_dir, f"02.{Country_name}_MAPSPAM_Crops", "02.Reprojected_Rasters" )
spam_filtered_dir, spam_summary_csv, spam_summary_df = Raster.mapspam_filter_crops(
    Country_name=Country_name,
    reprojected_dir=spam_reproj_dir,
    out_nodata=OUT_NODATA,
)

print("Filtered rasters:", spam_filtered_dir)
print("Summary CSV:", spam_summary_csv)

### Step 5 – Generate Density Rasters

The density rasters are calculated as harvsted area over total pixel area. They are needed in order to be resamples on the cropland point layer. 

In [ ]:
spam_filtered_dir =  os.path.join(Data_dir, f"02.{Country_name}_MAPSPAM_Crops", f"03.{Country_name}_Filtered_Rasters" )
spam_density_dir, density_files = Raster.calculate_density_rasters(
    Country_name=Country_name,
    filtered_dir=spam_filtered_dir,
    out_nodata=OUT_NODATA,
)

print("Density rasters:", spam_density_dir)
print(f"Created {len(density_files)} density rasters")

### Step 6 – Resample the rasters to the cropland point layer.

In [ ]:
#creates a GeoPackage with point geometries corresponding to the cropland locations
spam_density_dir = os.path.join(Data_dir, f"02.{Country_name}_MAPSPAM_Crops", "04.Density_Rasters" )
spam_final_dir, spam_final_fp, spam_final_gdf = Map.resample_raster_to_points(
    Country_name=Country_name,
    input_dir=spam_density_dir,
    cropland_gpkg=os.path.join(Data_dir, "01.BaseCroplandLayer", f"{Country_name}_crp_farmland.gpkg"),
)

print("Final resampled layer:", spam_final_fp)
print("Saved to folder:", spam_final_dir)

In [ ]:
# If Calibration_method is performed at sub-national level (adm1 or adm2), then compute the irrigation statistics per crop and admin unit. 
if Calibration_method > 0:
    admin_summary = Map.calculate_admin_mapspam_summary(
        Country_name=Country_name,
        input_gpkg=spam_final_fp,
    )

else:
    pass


In [ ]:
#This steps checks if there are any rows with a density higher than 1, which means that the harvested area is grater than the total pixel area. 

# spam_final_gdf = os.path.join(
#         Data_dir,
#         "03.Final",
#         f"01.{Country_name}_crop_IR",
#         f"{Country_name}_crop_IR.gpkg",
#     )
# Pixel_sum= Map.calculate_pixel_sum(Country_name=Country_name, input_gpkg=spam_final_gdf)
# print(Pixel_sum)

### Step 7 – Calibrate cropland

In [ ]:
#Calculate the total harvested area for each crop by summing the harvested area (calculated as density*pixel area) values across coordinate points in the GeoPackage.
spam_final_gdf = os.path.join(
        Data_dir,
        "03.Final",
        f"01.{Country_name}_crop_IR",
        f"{Country_name}_crop_IR.gpkg",
    )
mapspam_summary_csv = Map.mapspam_total_harvested_area(Country_name=Country_name, input_gpkg=spam_final_gdf)

print ("MAPSPAM summary CSV:", mapspam_summary_csv)

In [ ]:
#Calculate the national harvested area for each crop. This function reads the FAOSTAT table and filters by the user-defined country name.
National_harvested_area = Map.calculate_national_statistics(
    Country_name=Country_name,
    faostat_country_name=FAOSTAT_country_name,
    harvested_csv=National_statistics,
)
print("National harvested area per crop:", National_harvested_area)

In [ ]:
#If there is a mismatch between the total harvested area calculated from the GeoPackage and the national statistics, a scaling factor is calculated to adjust the values in the GeoPackage.
Scaling_factor = Map.calculate_scaling_factor(Country_name=Country_name)
print("Scaling factors for irrigated and rainfed crops:", Scaling_factor)

In [ ]:
#Here we create a gpk file that multiplies per each row the crop density values with the scaling factor.
calibrated_gpkg = Map.calibrate_crops(Country_name)
print("Calibrated GeoPackage:", calibrated_gpkg)

#We need to check if the sum of each row is > 1. Having a sum > 1 means that the total density (Total hatvested area/total available area) is grater than 1
#This has no phisical meaning. The .csv file returns all the rows with sum > 1. 
pixel_sum_result = Map.calculate_pixel_sum(Country_name, calibrated_gpkg)
print("Pixel sum result:", pixel_sum_result)

#### Re-calibrate crop density


When going from a lower resolution, in this case 10x10 km, to a higher one, the density of the crops can change. What can happen is that there might be rows with a total density higher than 1, which has no physical meaning since it means that the harvested area is higher than the total available area. The algorithm preserves the original crop composition of excess density from overloaded points and transfers it to the nearest available geographic points with remaining capacity, which means with density lower than 1.It is a composition-preserving redistribution algorithm. It treats each point's crop vector as a single package, transferring excess density proportionally across all crops to the nearest locations with available capacity.

In [ ]:
# Redistribute any overloaded crop densities so every point sum is <= 1.
calibrated2_gpkg = Map.redistribute_crop_density(
    Country_name=Country_name,
    target_crs=crs_proj,
)
print("Redistributed GeoPackage:", calibrated2_gpkg)

In [ ]:
calibrated2_gpkg = os.path.join(Data_dir, "03.Final", f"01.{Country_name}_crop_IR", f"{Country_name}_crop_IR_calibrated2.gpkg")

In [ ]:
# Validate the redistributed file: no row should exceed a total density of 1.
pixel_sum_check = Map.calculate_pixel_sum(Country_name, calibrated2_gpkg)
print("Pixel sum validation result:", pixel_sum_check)

In [ ]:
check_harvested_area = Map.mapspam_total_harvested_area(Country_name=Country_name, input_gpkg=calibrated2_gpkg)
print(check_harvested_area)

### Step 8 – Assign Farming Systems

This step reclassifies harvested area based on the four main farming systems: HI (high-irrigated), LI (low-irrigated), HR (high-rainfed), LR (low-rainfed). The value corresponds to the hectares (ha) of harvested area. Step 8 returns 1 cropland.gpkg whith all the crops and 1 irrigated.gpkg which extrapolates irrigated-only crops. Three CSV files with final statistics are created: commercial farming share, irrigation share, and final statistics that summarize for each crop the total harvested area in hectares on a national scale. All the data are stored in folder Data/03.Final.

In [ ]:
final_gpkg = Map.create_final_cropland_map(Country_name= Country_name)
print(final_gpkg)


In [ ]:
stats = Map.export_final_cropland_statistics(Country_name=  Country_name)
print(stats)

In [ ]:
# Create irrigated-only GeoPackage (keeps columns ending with 'I')
irrigated_gpkg = Map.create_irrigated_gpkg(Country_name=Country_name)
print('Irrigated GeoPackage:', irrigated_gpkg)

In [ ]:
import time
elapsed = time.perf_counter() - notebook_start
print(f"The workflow took: {elapsed:.2f} seconds")

### Step 9 – Add crop water demand

This final step estimates total crop water demand. Crop water deficit data are sourced from GAEZ v5. The dataset is provided in mm at approximately 10 km spatial resolution. Depending on user settings, the dataset can be sourced from different climate models. 

In [ ]:
#create a list of all the irrigated crops

Irrigated_cropsgpkg = os.path.join(Data_dir, '03.Final', f'{Country_name}_irrigated.gpkg')

irrigated_crops = [c for c in gpd.read_file(Irrigated_cropsgpkg).columns.tolist() if c not in ('x', 'y', 'geometry')]
print("The irrigated crops are:", irrigated_crops)

In [ ]:
#rename the crops to match the GAEZ naming convention. 
irrigated_crops_GAEZ = Water.convert_crops_to_GAEZ(irrigated_crop_list=irrigated_crops)
print("The irrigated crops converted to GAEZ are:", irrigated_crops_GAEZ)

In [ ]:
# Build GAEZ CWD URLs and download the files
cwd_input_dir = os.path.join(Input_data, "GAEZ_cwd")
url_records = Water.build_gaez_cwd_urls(
    SSP=SSP,
    Climate_model=Climate_model,
    irrigated_crops_GAEZ=irrigated_crops_GAEZ,
)
GAEZ_data = Water.download_gaez_rasters(url_records, cwd_input_dir)
print("Download summary:", GAEZ_data)

In [ ]:
#Clip cwd GAEZ rasters
cwd_input_dir = os.path.join(Input_data, "GAEZ_cwd")
output_cwd = os.path.join(Data_dir, '04.CropWaterDemand') 
country_boundary = os.path.join(Input_data, f"gadm41_{Country_name}_0.shp")

cwd_clip_dir, clipped_files = Raster.clip_rasters(
    Country_name=Country_name,
    input_dir=cwd_input_dir,
    boundary_path=country_boundary,
    output_dir=output_cwd
)

print("Clipped rasters:", cwd_clip_dir)

In [ ]:
#Reproject cwd rasters
cwd_clip_dir = os.path.join(Data_dir, '04.CropWaterDemand', f"01.{Country_name}_Clipped_Rasters")
cwd_output_reproj_dir = os.path.join(Data_dir, '04.CropWaterDemand', f"02.Reprojected_Rasters")
cwd_reproj_dir, reprojected_files = Raster.reproject_rasters(
    Country_name=Country_name,
    input_dir=cwd_clip_dir,
    output_dir=cwd_output_reproj_dir,
    out_nodata=OUT_NODATA,
    target_crs=crs_proj, 
)

print("Reprojected rasters:", cwd_reproj_dir)

In [ ]:
#resempling rasters to irrigation point layers.
cwd_rasters = os.path.join(Data_dir, '04.CropWaterDemand', f"02.Reprojected_Rasters")
cropland_gpkg = os.path.join(Data_dir, "03.Final", f"{Country_name}_irrigated.gpkg")
cwd_output_dir = os.path.join(Data_dir, '04.CropWaterDemand')
cwd_gpkg = Water.resample_cwd_to_points(Country_name= Country_name)

In [ ]:
import time
elapsed = time.perf_counter() - notebook_start
print(f"The workflow took: {elapsed:.2f} seconds")